In [ ]:
from pathlib import Path
import sys
import os

current_directory = Path(os.getcwd())
print(current_directory)
root_directory = current_directory.parent.parent
print(root_directory)

sys.path.append(str(root_directory))

In [ ]:
from pathlib import Path
import typing as t
import re
from pprint import pprint
import coq_serapy as c
import json
import csv
import pickle
import random

from src.config import CONFIG
from src.coq_serapy_util import LemmaLocation, read_commands, get_section_name_from_command, get_lemmas_from_commands
from src.dataset import Example

In [ ]:
PROJECT_DIR = Path(CONFIG.ROOT_DIR) / "coq-projects/coq-bb5"
PROJECT_DIR


In [ ]:
THEOREM_TOKENS = ["Theorem", "Lemma", "Fact", "Remark", "Corollary", "Proposition", "Property"]

In [ ]:
command = f"cd {PROJECT_DIR} && find . -name '*.v' -exec grep -o -E '{'|'.join(THEOREM_TOKENS)}' {{}} + | wc -l"
num_theorem_commands = int(os.popen(command).read())
num_theorem_commands

In [ ]:
command = f"cd {PROJECT_DIR} && find . -name '*.v' -exec grep -o 'Qed.' {{}} + | wc -l"
num_qeds = int(os.popen(command).read())
num_qeds

In [ ]:
command = f"cd {PROJECT_DIR} && find . -name '*.v' -exec grep -o 'Admitted.' {{}} + | wc -l"
num_admitted = int(os.popen(command).read())
num_admitted

In [ ]:
command = f"cd {PROJECT_DIR} && find . -name '*.v' -exec grep -o 'Defined.' {{}} + | wc -l"
num_defined = int(os.popen(command).read())
num_defined

In [ ]:
command = f"cd {PROJECT_DIR} && find . -name '*.v' -exec grep -o 'Proof.' {{}} + | wc -l"
num_proofs = int(os.popen(command).read())
num_proofs

In [ ]:
FILENAMES_TO_EXCLUDE = [
    "BB42Theorem",
    "BB52Theorem",
    "BB24Theorem",
    "Skelet1"
]

In [ ]:
COQ_PROJECT_PATH = PROJECT_DIR / "_CoqProject"
# filenames = all files after the first
filenames = [
    line.strip()
    for line in COQ_PROJECT_PATH.read_text().split("\n")[1:] 
    if line \
        and not line.strip().startswith("#") \
        and not any(f"{filename}.v" in line.strip() for filename in FILENAMES_TO_EXCLUDE)
]
filenames

In [ ]:
file_paths = [PROJECT_DIR / filename for filename in filenames]
file_paths

In [ ]:
file_to_num_theorem_commands: t.Dict[Path, int] = {
    file_path: int(os.popen(f"grep -o -E '{'|'.join(THEOREM_TOKENS)}' {file_path} | wc -l").read())
    for file_path in file_paths
}
file_to_num_theorem_commands


In [ ]:
file_to_num_qed: t.Dict[Path, int] = {
    file_path: int(os.popen(f"grep -o 'Qed.' {file_path} | wc -l").read())
    for file_path in file_paths
}
file_to_num_qed

In [ ]:
file_to_num_admitted: t.Dict[Path, int] = {
    file_path: int(os.popen(f"grep -o 'Admitted.' {file_path} | wc -l").read())
    for file_path in file_paths
}
file_to_num_admitted

In [ ]:
print(f"total_num_theorem_commands: {sum(file_to_num_theorem_commands.values())}")
print(f"total_num_qeds: {sum(file_to_num_qed.values())}")
print(f"total_num_admitted: {sum(file_to_num_admitted.values())}")


# get the nth proof in a file

In [ ]:
TEST_FILE = file_paths[1]
TEST_NUM_THEOREM_COMMANDS = file_to_num_theorem_commands[TEST_FILE]
TEST_NUM_QED = file_to_num_qed[TEST_FILE]
TEST_FILE, TEST_NUM_THEOREM_COMMANDS, TEST_NUM_QED

In [ ]:
TEST_IDX = 5

In [ ]:
TEST_FILE_COMMANDS = read_commands(TEST_FILE.read_text())
TEST_FILE_COMMANDS[0:5]

In [ ]:
num_lemmas = 0
for command in TEST_FILE_COMMANDS:
    try:
        lemma_name = c.coq_util.lemma_name_from_statement(command[0])
        if lemma_name.strip() == "":
            continue
        if not any(token in command[0] for token in THEOREM_TOKENS):
            continue
        num_lemmas += 1
        print(lemma_name, num_lemmas)
    except:
        continue

In [ ]:
lemmas, lemma_commands = get_lemmas_from_commands(
    project_name="coq-bb5",
    file_name=TEST_FILE.name,
    coq_version="8.18",
    commands=TEST_FILE_COMMANDS,
)

In [ ]:
lemmas

In [ ]:
len(lemmas)

In [ ]:
examples = [
    Example(
        location=lemma,
        proposition_command=command,
        gold_standard_proof="",
    )
    for lemma, command in zip(lemmas, lemma_commands)
]
examples

In [ ]:
len(examples)

In [ ]:
LEMMAS: t.Dict[Path, t.Tuple[t.List[LemmaLocation], t.List[str]]] = {
    file: get_lemmas_from_commands(
        project_name="coq-bb5",
        file_name=file.name,
        coq_version="8.18",
        commands=read_commands(file.read_text()),
    )
    for file in file_paths
}
LEMMAS

In [ ]:
EXAMPLES = {
    file: [
        Example(
            location=lemma,
            proposition_command=command,
            gold_standard_proof="",
        )
        for lemma, command in zip(lemmas, lemma_commands)
    ]
    for file, (lemmas, lemma_commands) in LEMMAS.items()
}
EXAMPLES

In [ ]:
NUM_EXAMPLES = {
    file: len(examples)
    for file, examples in EXAMPLES.items()
}
NUM_EXAMPLES
# Sample lemmas

In [ ]:

TOTAL_EXAMPLES = sum(NUM_EXAMPLES.values())
TOTAL_EXAMPLES

# Sample lemmas

In [ ]:
from src.dataset.coq_bb5 import COQ_BB5_DEV_SAMPLED_FILE, COQ_BB5_TEST_SAMPLED_FILE, COQ_BB5_DEV_CSV_FILE, COQ_BB5_TEST_CSV_FILE


In [ ]:
def sample(n: int, examples: t.Dict[Path, t.List[Example]]) -> t.List[Example]:
    concatenated_examples = [
        example
        for _, exs in examples.items()
        for example in exs
    ]

    return random.sample(concatenated_examples, n)

In [ ]:
TEST = sample(10, EXAMPLES)
TEST

In [ ]:
if not COQ_BB5_DEV_SAMPLED_FILE.exists():
    DEV = sorted(sample(10, EXAMPLES), key=lambda x: x.name)
    with COQ_BB5_DEV_SAMPLED_FILE.open("wb") as f:
        pickle.dump(DEV, f)
    DEV_EXAMPLE_NAMES = [example.name for example in DEV]
    with COQ_BB5_DEV_CSV_FILE.open("w") as f:
        writer = csv.writer(f)
        writer.writerow(["example_name"])
        for example_name in DEV_EXAMPLE_NAMES:
            writer.writerow([example_name])

In [ ]:
DEV = pickle.load(COQ_BB5_DEV_SAMPLED_FILE.open("rb"))
DEV


In [ ]:
EXAMPLES_NOT_IN_DEV = {
    file: [example for example in examples if example not in DEV]
    for file, examples in EXAMPLES.items()
}
EXAMPLES_NOT_IN_DEV

In [ ]:
TOTAL_EXAMPLES_NOT_IN_DEV = sum(len(examples) for examples in EXAMPLES_NOT_IN_DEV.values())
TOTAL_EXAMPLES_NOT_IN_DEV

In [ ]:
TOTAL_EXAMPLES

In [ ]:
if not COQ_BB5_TEST_SAMPLED_FILE.exists():
    TEST = sorted(sample(100, EXAMPLES_NOT_IN_DEV), key=lambda x: x.name)
    with COQ_BB5_TEST_SAMPLED_FILE.open("wb") as f:
        pickle.dump(TEST, f)
    TEST_EXAMPLE_NAMES = [example.name for example in TEST]
    with COQ_BB5_TEST_CSV_FILE.open("w") as f:
        writer = csv.writer(f)
        writer.writerow(["example_name"])
        for example_name in TEST_EXAMPLE_NAMES:
            writer.writerow([example_name])

In [ ]:
TEST = pickle.load(COQ_BB5_TEST_SAMPLED_FILE.open("rb"))
TEST


# remove proofs not in coqstoq and sample new ones

In [ ]:
EXAMPLES_NOT_IN_COQSTOQ = [
    'coq-bb5-Skelet17.v-Increments_precond_toConfig',
    "coq-bb5-Skelet17.v-N'steps_precond_toConfig",
    "coq-bb5-Skelet17.v-Nat_leb_S_l",
    "coq-bb5-Skelet17.v-OZIH_toConfig",
    "coq-bb5-Skelet17.v-ZIHIO_emb_Add2",
    "coq-bb5-Skelet17.v-ctzS_even_mod4ne1",
    "coq-bb5-Skelet17.v-ctzS_spec_0",
    "coq-bb5-Skelet17.v-embanked_batches",
] + [
    "coq-bb5-Skelet17.v-embanked__embanked_batch",
    "coq-bb5-Skelet17.v-embanked_batch_Add2s"
]
len(EXAMPLES_NOT_IN_COQSTOQ)


In [ ]:
TEST_EXAMPLE_NAMES = [example.name for example in TEST]
all(example_name in TEST_EXAMPLE_NAMES for example_name in EXAMPLES_NOT_IN_COQSTOQ)

In [ ]:
TEST_EXAMPLES_IN_COQSTOQ = [example for example in TEST if example.name not in EXAMPLES_NOT_IN_COQSTOQ]
len(TEST_EXAMPLES_IN_COQSTOQ)

In [ ]:
new_examples = sample(30, EXAMPLES)
new_examples

In [ ]:
new_examples = [example for example in new_examples if example.name not in EXAMPLES_NOT_IN_COQSTOQ]
len(new_examples)

In [ ]:
new_examples = [example for example in new_examples if example not in TEST_EXAMPLES_IN_COQSTOQ]
len(new_examples)

In [ ]:
new_examples = [example for example in new_examples if example not in DEV]
len(new_examples)


In [ ]:
num_examples_to_add = 100 - len(TEST_EXAMPLES_IN_COQSTOQ)
print(f"num_examples_to_add: {num_examples_to_add}")
extra_examples_to_add = 0

examples_to_add = new_examples[:num_examples_to_add + extra_examples_to_add]
list(map(lambda x: x.name, sorted(examples_to_add, key=lambda x: x.name)))



In [ ]:
BAD_EXAMPLE_NAMES = [
]

examples_to_add = [example for example in examples_to_add if example.name not in BAD_EXAMPLE_NAMES]
print(len(examples_to_add))
list(map(lambda x: x.name, sorted(examples_to_add, key=lambda x: x.name)))

In [ ]:
UPDATED_TEST = TEST_EXAMPLES_IN_COQSTOQ + examples_to_add
len(UPDATED_TEST)

In [ ]:
OVERWRITE_TEST_FILE = False


In [ ]:
if OVERWRITE_TEST_FILE:
    with COQ_BB5_TEST_SAMPLED_FILE.open("wb") as f:
        pickle.dump(UPDATED_TEST, f)
    TEST_EXAMPLE_NAMES = [example.name for example in UPDATED_TEST]
    with COQ_BB5_TEST_CSV_FILE.open("w") as f:
        writer = csv.writer(f)
        writer.writerow(["example_name"])
        for example_name in TEST_EXAMPLE_NAMES:
            writer.writerow([example_name])